# News Headline Generation - analysis walkthrough
Run from the project root. Edit the three settings below.

In [ ]:
import sys; sys.path.insert(0,'src')
import pandas as pd, matplotlib.pyplot as plt
from preprocessing import *
from model import *
from evaluation import evaluate
DATA, TEXT, HEAD = 'dataset/news.csv', 'Short', 'Headline'

## 1. Load, clean, explore

In [ ]:
df = load_dataset(DATA, TEXT, HEAD)
print(len(df)); df.head()

In [ ]:
df.assign(article_words=df.text.str.split().str.len(), headline_words=df.headline.str.split().str.len()).describe()

## 2. Preprocessing demo (tokenise -> stop words -> lemmas)

In [ ]:
s = df.text[0]
toks = tokenize(s); print(toks[:15]); print(normalize_tokens(toks)[:15])

## 3. Split, fit TF-IDF, generate

In [ ]:
train, val, test = split_dataset(df)
vec = fit_vectorizer(train.text)
gens = [LeadHeadline(), TfidfHeadline(vec), TfidfHeadline(vec, 1.0), TextRankHeadline(vec)]
sample = test.head(5)
for t, h in zip(sample.text, sample.headline):
    print('REF:', h)
    for g in gens: print(f'  {g.name:28s}', g.generate(t))

## 4. Evaluate (ROUGE)

In [ ]:
res = pd.DataFrame({g.name: evaluate(test.headline, [g.generate(t) for t in test.text]) for g in gens}).T
res

In [ ]:
res[['ROUGE-1 F1','ROUGE-2 F1','ROUGE-L F1']].plot.bar(rot=15); plt.show()

## 5. Interpretation
Write your own conclusions here: which approach wins, why, and where it fails (see outputs/error_analysis.csv).